# Backtest NQ Futures with EMACrossTWAP Strategy

Tutorial for [NautilusTrader](https://nautilustrader.io/docs/) demonstrating backtesting the EMACrossTWAP strategy on NQ futures using 1-minute bar data.

[View source on GitHub](https://github.com/nautechsystems/nautilus_trader/blob/develop/docs/getting_started/backtest_low_level_futures_NQ.ipynb).

## Overview

This tutorial demonstrates how to backtest the EMACrossTWAP strategy on NQ futures contracts using NautilusTrader's low-level API.

The following points will be covered:
- How to create an NQ futures contract instrument
- How to load 1-minute NQ futures data
- How to configure the EMACrossTWAP strategy for futures trading
- How to set up a futures trading venue
- How to run backtests and analyze results

## Prerequisites
- Python 3.11+ installed
- [JupyterLab](https://jupyter.org/) or similar installed (`pip install -U jupyterlab`)
- [NautilusTrader](https://pypi.org/project/nautilus_trader/) latest release installed (`pip install -U nautilus_trader`)
- 1-minute NQ futures data (CSV format with columns: timestamp, open, high, low, close, volume)

## Imports

We'll start with all of our imports for the remainder of this tutorial.

In [ ]:
from decimal import Decimal
import pandas as pd
import pytz

from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.backtest.engine import BacktestEngineConfig
from nautilus_trader.examples.algorithms.twap import TWAPExecAlgorithm
from nautilus_trader.examples.strategies.ema_cross_twap import EMACrossTWAP
from nautilus_trader.examples.strategies.ema_cross_twap import EMACrossTWAPConfig
from nautilus_trader.model import BarType
from nautilus_trader.model import Money
from nautilus_trader.model import TraderId
from nautilus_trader.model import Venue
from nautilus_trader.model.currencies import USD
from nautilus_trader.model.enums import AccountType
from nautilus_trader.model.enums import AssetClass
from nautilus_trader.model.enums import OmsType
from nautilus_trader.model.identifiers import InstrumentId
from nautilus_trader.model.identifiers import Symbol
from nautilus_trader.model.instruments import FuturesContract
from nautilus_trader.model.objects import Price
from nautilus_trader.model.objects import Quantity
from nautilus_trader.persistence.wranglers import BarDataWrangler

## Create NQ Futures Instrument

First, we'll create an NQ futures contract instrument. NQ is the E-mini NASDAQ-100 futures contract traded on CME.

In [ ]:
# Create NQ futures contract (example: NQH4 - March 2024)
def create_nq_futures_contract(
    symbol: str = "NQH4",
    expiry_year: int = 2024,
    expiry_month: int = 3,
) -> FuturesContract:
    """
    Create an NQ futures contract.
    
    Parameters
    ----------
    symbol : str
        The futures contract symbol (e.g., 'NQH4')
    expiry_year : int
        The expiration year
    expiry_month : int
        The expiration month
    
    Returns
    -------
    FuturesContract
    """
    # Calculate activation and expiration dates
    # NQ futures typically expire on the third Friday of the contract month
    expiration_date = pd.Timestamp(f"{expiry_year}-{expiry_month:02d}-15", tz=pytz.utc)
    # Find the third Friday
    while expiration_date.weekday() != 4:  # Friday is 4
        expiration_date += pd.Timedelta(days=1)
    expiration_date += pd.Timedelta(days=14)  # Third Friday
    
    # Activation is typically 2 years before expiration
    activation_date = expiration_date - pd.DateOffset(years=2)
    
    return FuturesContract(
        instrument_id=InstrumentId(symbol=Symbol(symbol), venue=Venue("CME")),
        raw_symbol=Symbol(symbol),
        asset_class=AssetClass.INDEX,
        exchange="XCME",
        currency=USD,
        price_precision=2,
        price_increment=Price.from_str("0.25"),  # NQ minimum tick is 0.25 points
        multiplier=Quantity.from_int(20),  # NQ multiplier is $20 per point
        lot_size=Quantity.from_int(1),
        underlying="NQ",
        activation_ns=activation_date.value,
        expiration_ns=expiration_date.value,
        ts_event=activation_date.value,
        ts_init=activation_date.value,
    )

# Create the NQ futures contract
NQ_FUTURES = create_nq_futures_contract()

## Loading Data

For this tutorial, you'll need 1-minute NQ futures data. The data should be in CSV format with columns: timestamp, open, high, low, close, volume.

**Note**: Since no NQ data is included in the repository, you'll need to provide your own data file. Update the file path below to point to your NQ data file.

In [ ]:
# Load NQ futures data
# TODO: Replace with your actual NQ data file path
data_file_path = "path/to/your/nq_1min_data.csv"

try:
    # Read the CSV data
    df = pd.read_csv(data_file_path)
    
    # Ensure proper column names and data types
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    elif 'datetime' in df.columns:
        df['timestamp'] = pd.to_datetime(df['datetime'])
    else:
        # Assume first column is timestamp
        df.columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume']
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Set timezone to UTC if not already set
    if df['timestamp'].dt.tz is None:
        df['timestamp'] = df['timestamp'].dt.tz_localize('UTC')
    
    print(f"Loaded {len(df)} rows of NQ data")
    print(f"Data range: {df['timestamp'].min()} to {df['timestamp'].max()}")
    print(df.head())
    
except FileNotFoundError:
    print(f"Data file not found: {data_file_path}")
    print("Please provide 1-minute NQ futures data in CSV format.")
    print("Expected columns: timestamp, open, high, low, close, volume")
    
    # Create sample data for demonstration
    print("\nCreating sample data for demonstration...")
    import numpy as np
    
    # Generate sample 1-minute bars for 1 day
    start_time = pd.Timestamp("2024-01-15 09:30:00", tz="UTC")
    end_time = pd.Timestamp("2024-01-15 16:00:00", tz="UTC")
    timestamps = pd.date_range(start_time, end_time, freq="1min")
    
    # Generate realistic NQ price data (around 17000 level)
    np.random.seed(42)
    base_price = 17000.0
    price_changes = np.random.normal(0, 5, len(timestamps)).cumsum()
    
    opens = base_price + price_changes
    closes = opens + np.random.normal(0, 2, len(timestamps))
    highs = np.maximum(opens, closes) + np.abs(np.random.normal(0, 1, len(timestamps)))
    lows = np.minimum(opens, closes) - np.abs(np.random.normal(0, 1, len(timestamps)))
    volumes = np.random.randint(100, 1000, len(timestamps))
    
    df = pd.DataFrame({
        'timestamp': timestamps,
        'open': opens,
        'high': highs,
        'low': lows,
        'close': closes,
        'volume': volumes
    })
    
    print(f"Generated {len(df)} sample bars")
    print(df.head())

## Process Data into Nautilus Objects

Now we need to wrangle this data into Nautilus `Bar` objects that can be used by the backtest engine.

In [ ]:
# Create bar type for 1-minute bars
bar_type = BarType.from_str(f"{NQ_FUTURES.id}-1-MINUTE-LAST-EXTERNAL")

# Process data using BarDataWrangler
wrangler = BarDataWrangler(
    bar_type=bar_type,
    instrument=NQ_FUTURES,
)

# Convert DataFrame to Nautilus bars
bars = wrangler.process(df)

print(f"Processed {len(bars)} bars")
print(f"First bar: {bars[0]}")
print(f"Last bar: {bars[-1]}")

## Initialize Backtest Engine

Now we'll create a backtest engine with appropriate configuration for futures trading.

In [ ]:
# Configure backtest engine
config = BacktestEngineConfig(
    trader_id=TraderId("BACKTESTER-NQ-001"),
    log_level="INFO",
)

# Build the backtest engine
engine = BacktestEngine(config=config)
print("Backtest engine initialized")

## Add Futures Venue

We'll set up a simulated CME venue for futures trading with appropriate margin requirements.

In [ ]:
# Add CME futures venue
CME = Venue("CME")
engine.add_venue(
    venue=CME,
    oms_type=OmsType.NETTING,
    account_type=AccountType.MARGIN,  # Futures require margin account
    base_currency=USD,
    starting_balances=[Money(100_000.0, USD)],  # $100k starting capital
)
print("CME venue added with $100,000 starting capital")

## Add Instrument and Data

Add the NQ futures contract and the processed bar data to the backtest engine.

In [ ]:
# Add the NQ futures instrument
engine.add_instrument(NQ_FUTURES)
print(f"Added instrument: {NQ_FUTURES.id}")

# Add the bar data
engine.add_data(bars)
print(f"Added {len(bars)} bars to the engine")

## Configure EMACrossTWAP Strategy

Now we'll configure the EMACrossTWAP strategy specifically for NQ futures trading.

In [ ]:
# Configure the EMACrossTWAP strategy for NQ futures
strategy_config = EMACrossTWAPConfig(
    instrument_id=NQ_FUTURES.id,
    bar_type=bar_type,
    trade_size=Decimal("1"),  # 1 contract per trade
    fast_ema_period=10,       # 10-period fast EMA
    slow_ema_period=20,       # 20-period slow EMA
    twap_horizon_secs=300.0,  # 5-minute TWAP horizon
    twap_interval_secs=60.0,  # 1-minute intervals between TWAP orders
)

# Instantiate the strategy
strategy = EMACrossTWAP(config=strategy_config)
engine.add_strategy(strategy=strategy)
print("EMACrossTWAP strategy configured and added")
print(f"Trade size: {strategy_config.trade_size} contracts")
print(f"Fast EMA: {strategy_config.fast_ema_period} periods")
print(f"Slow EMA: {strategy_config.slow_ema_period} periods")
print(f"TWAP horizon: {strategy_config.twap_horizon_secs} seconds")

## Add TWAP Execution Algorithm

Add the TWAP execution algorithm that will handle order execution.

In [ ]:
# Add TWAP execution algorithm
exec_algorithm = TWAPExecAlgorithm()
engine.add_exec_algorithm(exec_algorithm)
print("TWAP execution algorithm added")

## Run Backtest

Now we can run the backtest over all available data.

In [ ]:
# Run the backtest
print("Starting backtest...")
engine.run()
print("Backtest completed!")

## Post-Run Analysis

After the backtest completes, we can analyze the results and generate reports.

In [ ]:
# Generate account report
account_report = engine.trader.generate_account_report(CME)
print("\n=== ACCOUNT REPORT ===")
print(account_report.tail(10))  # Show last 10 entries

In [ ]:
# Generate order report
order_report = engine.trader.generate_order_fills_report()
print("\n=== ORDER FILLS REPORT ===")
print(f"Total orders: {len(order_report)}")
if len(order_report) > 0:
    print(order_report[['instrument_id', 'side', 'quantity', 'avg_px', 'commission']].head(10))

In [ ]:
# Generate position report
position_report = engine.trader.generate_positions_report()
print("\n=== POSITIONS REPORT ===")
print(f"Total positions: {len(position_report)}")
if len(position_report) > 0:
    print(position_report[['instrument_id', 'side', 'quantity', 'avg_px_open', 'realized_pnl']].head(10))

In [ ]:
# Calculate basic performance metrics
portfolio = engine.trader.portfolio
account = portfolio.account(CME)

print("\n=== PERFORMANCE SUMMARY ===")
print(f"Starting balance: $100,000.00")
print(f"Ending balance: ${account.balance_total(USD):.2f}")
print(f"Total PnL: ${account.balance_total(USD) - 100_000:.2f}")
print(f"Total return: {((account.balance_total(USD) / 100_000) - 1) * 100:.2f}%")

# Get realized PnL
realized_pnl = sum(pos.realized_pnl.as_double() for pos in portfolio.positions_closed())
print(f"Realized PnL: ${realized_pnl:.2f}")

# Get unrealized PnL
unrealized_pnl = sum(pos.unrealized_pnl(pos.last_px).as_double() for pos in portfolio.positions_open())
print(f"Unrealized PnL: ${unrealized_pnl:.2f}")

print(f"\nTotal trades: {len(portfolio.positions())}")
print(f"Open positions: {len(portfolio.positions_open())}")
print(f"Closed positions: {len(portfolio.positions_closed())}")

## Next Steps

This notebook demonstrates the basic setup for backtesting NQ futures with the EMACrossTWAP strategy. You can extend this by:

1. **Adding more sophisticated data**: Use tick data or higher frequency bars
2. **Parameter optimization**: Test different EMA periods and TWAP settings
3. **Risk management**: Add position sizing rules and stop losses
4. **Multiple timeframes**: Combine different bar types for signal generation
5. **Portfolio analysis**: Add more detailed performance metrics and visualizations
6. **Live trading**: Adapt the strategy for live trading with real market data

Remember to:
- Use realistic transaction costs and slippage
- Consider margin requirements for futures trading
- Test with out-of-sample data
- Validate results with different market conditions